# NBA Live Win Probability — Dataset Builder

This notebook builds the dataset used for a live NBA win-probability model.

Pipeline:

1. Pull NBA live play-by-play JSON files from `cdn.nba.com`
2. Convert raw actions into game-state rows
3. Add score, clock, possession, foul, timeout, shot, and player/action fields
4. Compute team rolling strength features from prior games only
5. Compute rest/back-to-back schedule features
6. Join everything into full and slim modeling tables

The goal is to keep this notebook reproducible, readable, and GitHub-friendly.  
Large CSV outputs are intentionally written to `data/processed/` and should usually be ignored by Git unless sampled.

## 0. Setup

Run the install cell only if your environment does not already have the required packages.

In [ ]:
# Optional install if running in a fresh environment
# %pip install pandas numpy requests tqdm pyarrow

In [1]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path
from typing import Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

c:\Users\joshu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

Change `SEASONS` to control which NBA seasons are pulled.  
For example, `2024` means the 2024-25 season.

Set `RUN_DATA_PULL = False` if you already have `nba_win_prob_dataset.csv` and only want to rebuild derived files.

In [2]:
# Seasons to collect. Example: 2022 -> 2022-23 NBA season.
SEASONS = [2022, 2023, 2024, 2025]

RUN_DATA_PULL = True

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

BASE_PBP_PATH = PROCESSED_DIR / "nba_win_prob_dataset.csv"
TEAM_STATS_PATH = PROCESSED_DIR / "team_rolling_stats.csv"
REST_PATH = PROCESSED_DIR / "game_rest_features.csv"
FULL_OUTPUT_PATH = PROCESSED_DIR / "nba_win_prob_full_enriched.csv"
SLIM_OUTPUT_PATH = PROCESSED_DIR / "nba_win_prob_slim.csv"

REQUEST_SLEEP_SECONDS = 0.50
CHECKPOINT_EVERY_GAMES = 100

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Referer": "https://www.nba.com/",
}

## 2. NBA team mapping and API helpers

In [3]:
TEAM_ID_TO_TRICODE = {
    1610612737: "ATL", 1610612738: "BOS", 1610612751: "BKN", 1610612766: "CHA",
    1610612741: "CHI", 1610612739: "CLE", 1610612742: "DAL", 1610612743: "DEN",
    1610612765: "DET", 1610612744: "GSW", 1610612745: "HOU", 1610612754: "IND",
    1610612746: "LAC", 1610612747: "LAL", 1610612763: "MEM", 1610612748: "MIA",
    1610612749: "MIL", 1610612750: "MIN", 1610612740: "NOP", 1610612752: "NYK",
    1610612760: "OKC", 1610612753: "ORL", 1610612755: "PHI", 1610612756: "PHX",
    1610612757: "POR", 1610612758: "SAC", 1610612759: "SAS", 1610612761: "TOR",
    1610612762: "UTA", 1610612764: "WAS",
}
TRICODE_TO_TEAM_ID = {v: k for k, v in TEAM_ID_TO_TRICODE.items()}


def generate_regular_season_ids(season_year: int, max_games: int = 1300) -> List[str]:
    """Generate possible regular-season game IDs for one NBA season."""
    short_year = str(season_year + 1)[-2:]
    return [f"002{short_year}{i:05d}" for i in range(1, max_games + 1)]


def generate_playoff_ids(season_year: int) -> List[str]:
    """Generate likely NBA playoff game IDs."""
    short_year = str(season_year + 1)[-2:]

    series_codes = [
        10, 11, 12, 13, 14, 15, 16, 17,  # round 1
        20, 21, 22, 23,                  # conference semis
        30, 31,                          # conference finals
        40,                              # finals
    ]

    return [f"004{short_year}00{series_code}{game_num}"
            for series_code in series_codes
            for game_num in range(1, 8)]


def generate_game_ids(seasons: Iterable[int], include_playoffs: bool = True) -> List[Tuple[str, bool]]:
    """Return tuples of (game_id, is_playoffs)."""
    ids = []

    for season_year in seasons:
        ids.extend((gid, False) for gid in generate_regular_season_ids(season_year))

        if include_playoffs:
            ids.extend((gid, True) for gid in generate_playoff_ids(season_year))

    seen = set()
    unique_ids = []
    for gid, is_playoffs in ids:
        if gid not in seen:
            seen.add(gid)
            unique_ids.append((gid, is_playoffs))

    return unique_ids


def fetch_pbp_json(game_id: str, timeout: int = 60) -> Optional[dict]:
    """Fetch one NBA live play-by-play JSON file."""
    url = f"https://cdn.nba.com/static/json/liveData/playbyplay/playbyplay_{game_id}.json"

    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        if response.status_code != 200:
            return None

        payload = response.json()
        if "game" not in payload or "actions" not in payload["game"]:
            return None

        if not payload["game"]["actions"]:
            return None

        return payload

    except Exception:
        return None

## 3. Raw play-by-play → game-state rows

The model predicts `home_team_won`, so every action row is converted into a live game state from the home team's perspective.

In [4]:
def parse_iso_clock(clock_str) -> Optional[float]:
    """Convert NBA ISO-ish clock strings like 'PT11M42.00S' to seconds remaining in period."""
    match = re.match(r"PT(\d+)M([\d.]+)S", str(clock_str))
    if not match:
        return np.nan

    minutes = int(match.group(1))
    seconds = float(match.group(2))
    return minutes * 60 + seconds


def extract_game_date(game_info: dict) -> pd.Timestamp:
    """Best-effort game date extraction from NBA live JSON metadata."""
    date_candidates = [
        game_info.get("gameEt"),
        game_info.get("gameTimeUTC"),
        game_info.get("gameDateTimeUTC"),
        game_info.get("gameDate"),
        game_info.get("gameCode", "")[:8],
    ]

    for value in date_candidates:
        if value is None or value == "":
            continue

        parsed = pd.to_datetime(value, errors="coerce")
        if pd.notna(parsed):
            return parsed.normalize()

    return pd.NaT


def extract_home_away_ids(game_info: dict, actions: pd.DataFrame) -> Tuple[Optional[int], Optional[int]]:
    """Extract home/away team IDs from metadata, with a fallback using tricodes in actions."""
    home_team = game_info.get("homeTeam", {}) or {}
    away_team = game_info.get("awayTeam", {}) or {}

    home_id = home_team.get("teamId") or home_team.get("teamID")
    away_id = away_team.get("teamId") or away_team.get("teamID")

    if home_id is not None and away_id is not None:
        return int(home_id), int(away_id)

    home_tri = home_team.get("teamTricode") or home_team.get("tricode")
    away_tri = away_team.get("teamTricode") or away_team.get("tricode")

    if home_tri in TRICODE_TO_TEAM_ID and away_tri in TRICODE_TO_TEAM_ID:
        return TRICODE_TO_TEAM_ID[home_tri], TRICODE_TO_TEAM_ID[away_tri]

    teams = actions[["teamId", "teamTricode"]].dropna().drop_duplicates()

    if home_tri in set(teams["teamTricode"]):
        home_id = int(teams.loc[teams["teamTricode"] == home_tri, "teamId"].iloc[0])
    if away_tri in set(teams["teamTricode"]):
        away_id = int(teams.loc[teams["teamTricode"] == away_tri, "teamId"].iloc[0])

    return home_id, away_id


def normalize_qualifiers(value) -> str:
    """Convert NBA qualifier lists/dicts/strings into a lowercase searchable string."""
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                parts.extend(str(v) for v in item.values())
            else:
                parts.append(str(item))
        return " ".join(parts).lower()

    if isinstance(value, dict):
        return " ".join(str(v) for v in value.values()).lower()

    return str(value).lower()


def forward_fill_team_stat(df: pd.DataFrame, team_id: int, source_col: str) -> pd.Series:
    """Forward-fill a cumulative team stat from rows where the given team appears."""
    if source_col not in df.columns:
        return pd.Series(0, index=df.index)

    values = pd.to_numeric(df.loc[df["teamId"] == team_id, source_col], errors="coerce")
    return values.reindex(df.index).ffill().fillna(0)


def build_game_state(payload: dict, game_id: str, is_playoffs: bool) -> Optional[pd.DataFrame]:
    """Convert one raw NBA live JSON payload into cleaned game-state rows."""
    game_info = payload["game"]
    actions = pd.DataFrame(game_info["actions"])

    if actions.empty:
        return None

    home_team_id, away_team_id = extract_home_away_ids(game_info, actions)

    if home_team_id is None or away_team_id is None:
        return None

    home_tricode = TEAM_ID_TO_TRICODE.get(home_team_id)
    away_tricode = TEAM_ID_TO_TRICODE.get(away_team_id)
    game_date = extract_game_date(game_info)

    df = actions.copy()

    expected_cols = [
        "actionType", "subType", "clock", "period", "possession",
        "teamId", "teamTricode", "personId", "playerName", "description",
        "scoreHome", "scoreAway", "foulPersonalTotal", "foulTechnicalTotal",
        "shotDistance", "area", "areaDetail", "shotResult", "isFieldGoal",
        "x", "y", "qualifiers",
    ]
    for col in expected_cols:
        if col not in df.columns:
            df[col] = np.nan

    action_types_keep = [
        "2pt", "3pt", "freethrow", "turnover", "foul",
        "rebound", "block", "steal", "timeout", "ejection",
    ]
    df = df[df["actionType"].isin(action_types_keep)].copy()

    if df.empty:
        return None

    df["seconds_in_period"] = df["clock"].apply(parse_iso_clock)
    df["period"] = pd.to_numeric(df["period"], errors="coerce")

    # Regulation: total seconds remaining in the scheduled 48 minutes.
    # Overtime: seconds left in the current OT period; period carries the OT number.
    df["seconds_remaining"] = np.where(
        df["period"] <= 4,
        (4 - df["period"]) * 720 + df["seconds_in_period"],
        df["seconds_in_period"],
    )
    df["is_overtime"] = (df["period"] > 4).astype(int)

    df["scoreHome"] = pd.to_numeric(df["scoreHome"], errors="coerce").ffill().fillna(0)
    df["scoreAway"] = pd.to_numeric(df["scoreAway"], errors="coerce").ffill().fillna(0)
    df["score_diff"] = df["scoreHome"] - df["scoreAway"]

    df["momentum_5"] = df["score_diff"].diff(5)
    df["momentum_10"] = df["score_diff"].diff(10)

    df["possession"] = pd.to_numeric(df["possession"], errors="coerce")
    df["home_possession"] = (df["possession"] == home_team_id).astype(int)

    df["teamId"] = pd.to_numeric(df["teamId"], errors="coerce")
    df["home_fouls"] = forward_fill_team_stat(df, home_team_id, "foulPersonalTotal")
    df["away_fouls"] = forward_fill_team_stat(df, away_team_id, "foulPersonalTotal")
    df["home_tech_fouls"] = forward_fill_team_stat(df, home_team_id, "foulTechnicalTotal")
    df["away_tech_fouls"] = forward_fill_team_stat(df, away_team_id, "foulTechnicalTotal")

    df["home_ejections"] = ((df["actionType"] == "ejection") & (df["teamId"] == home_team_id)).cumsum()
    df["away_ejections"] = ((df["actionType"] == "ejection") & (df["teamId"] == away_team_id)).cumsum()

    df["home_timeouts_used"] = ((df["actionType"] == "timeout") & (df["teamId"] == home_team_id)).cumsum()
    df["away_timeouts_used"] = ((df["actionType"] == "timeout") & (df["teamId"] == away_team_id)).cumsum()

    df["shot_distance"] = pd.to_numeric(df["shotDistance"], errors="coerce")
    df["shot_area"] = df["area"]
    df["shot_area_detail"] = df["areaDetail"]
    df["shot_result"] = df["shotResult"]
    df["is_field_goal"] = pd.to_numeric(df["isFieldGoal"], errors="coerce").fillna(0).astype(int)
    df["shot_x"] = pd.to_numeric(df["x"], errors="coerce")
    df["shot_y"] = pd.to_numeric(df["y"], errors="coerce")

    qualifiers = df["qualifiers"].apply(normalize_qualifiers)
    df["is_fastbreak"] = qualifiers.str.contains("fastbreak", na=False).astype(int)
    df["is_painttouch"] = qualifiers.str.contains("paint|pointsinthepaint", na=False).astype(int)
    df["is_fromturnover"] = qualifiers.str.contains("turnover|fromturnover", na=False).astype(int)

    df["freethrow_subtype"] = np.where(df["actionType"] == "freethrow", df["subType"], None)
    df["person_id"] = df["personId"]
    df["player_name"] = df["playerName"]

    final_home = df["scoreHome"].dropna().iloc[-1]
    final_away = df["scoreAway"].dropna().iloc[-1]
    home_team_won = int(final_home > final_away)

    df["game_id"] = str(game_id).zfill(10)
    df["is_playoffs"] = int(is_playoffs)
    df["home_team_won"] = home_team_won
    df["home_team_id"] = home_team_id
    df["away_team_id"] = away_team_id
    df["home_tricode"] = home_tricode
    df["away_tricode"] = away_tricode
    df["game_date"] = game_date

    keep_cols = [
        "game_id", "is_playoffs", "home_team_won",
        "home_team_id", "away_team_id", "home_tricode", "away_tricode", "game_date",
        "period", "is_overtime", "seconds_remaining",
        "scoreHome", "scoreAway", "score_diff", "momentum_5", "momentum_10",
        "home_possession", "actionType", "teamTricode",
        "home_fouls", "away_fouls", "home_tech_fouls", "away_tech_fouls",
        "home_ejections", "away_ejections", "home_timeouts_used", "away_timeouts_used",
        "shot_distance", "shot_area", "shot_area_detail", "shot_result",
        "is_field_goal", "shot_x", "shot_y",
        "is_fastbreak", "is_painttouch", "is_fromturnover",
        "freethrow_subtype", "person_id", "player_name", "description",
    ]

    return df[keep_cols].reset_index(drop=True)

## 4. Pull and checkpoint play-by-play data

This writes checkpoints, so the job can resume if interrupted.

In [5]:
def collect_play_by_play(
    seasons: Iterable[int],
    output_path: Path = BASE_PBP_PATH,
    include_playoffs: bool = True,
    checkpoint_every_games: int = CHECKPOINT_EVERY_GAMES,
) -> pd.DataFrame:
    """Collect live play-by-play for many seasons and save a cleaned base dataset."""
    output_path = Path(output_path)
    all_frames = []
    processed_ids = set()

    if output_path.exists():
        existing = pd.read_csv(output_path, low_memory=False)
        existing["game_id"] = existing["game_id"].astype(str).str.zfill(10)
        processed_ids = set(existing["game_id"].unique())
        all_frames.append(existing)
        print(f"Resuming from {output_path}: {len(processed_ids):,} games already processed.")

    candidate_ids = generate_game_ids(seasons, include_playoffs=include_playoffs)
    candidate_ids = [(gid, is_po) for gid, is_po in candidate_ids if gid not in processed_ids]

    print(f"Candidate IDs to try: {len(candidate_ids):,}")

    games_added = 0
    games_missing = 0

    for game_id, is_playoffs in tqdm(candidate_ids):
        payload = fetch_pbp_json(game_id)

        if payload is None:
            games_missing += 1
            time.sleep(REQUEST_SLEEP_SECONDS)
            continue

        clean = build_game_state(payload, game_id, is_playoffs)

        if clean is None or clean.empty:
            games_missing += 1
            time.sleep(REQUEST_SLEEP_SECONDS)
            continue

        all_frames.append(clean)
        processed_ids.add(game_id)
        games_added += 1

        if games_added % checkpoint_every_games == 0:
            checkpoint = pd.concat(all_frames, ignore_index=True)
            checkpoint.to_csv(output_path, index=False)
            print(f"Checkpoint saved: {checkpoint['game_id'].nunique():,} games, {len(checkpoint):,} rows.")

        time.sleep(REQUEST_SLEEP_SECONDS)

    if not all_frames:
        raise RuntimeError("No play-by-play data collected.")

    result = pd.concat(all_frames, ignore_index=True)
    result["game_id"] = result["game_id"].astype(str).str.zfill(10)
    result.to_csv(output_path, index=False)

    print(f"Done: {result['game_id'].nunique():,} games, {len(result):,} rows.")
    print(f"Missing/nonexistent IDs skipped: {games_missing:,}")
    print(f"Saved to: {output_path}")

    return result


if RUN_DATA_PULL:
    pbp = collect_play_by_play(SEASONS, BASE_PBP_PATH, include_playoffs=True)
else:
    pbp = pd.read_csv(BASE_PBP_PATH, low_memory=False)
    pbp["game_id"] = pbp["game_id"].astype(str).str.zfill(10)
    print(f"Loaded existing base dataset: {pbp.shape}")

Candidate IDs to try: 5,620


  1%|          | 53/5620 [01:09<2:00:58,  1.30s/it]


KeyboardInterrupt: 

## 5. Rest and schedule features

Rest is computed directly from the games collected in the play-by-play dataset, which avoids maintaining a separate schedule file.

In [ ]:
def build_game_schedule_from_pbp(pbp: pd.DataFrame) -> pd.DataFrame:
    """Create one row per game with date and home/away tricodes."""
    cols = [
        "game_id", "game_date", "home_team_id", "away_team_id",
        "home_tricode", "away_tricode", "is_playoffs", "home_team_won",
    ]

    schedule = pbp[cols].drop_duplicates("game_id").copy()
    schedule["game_id"] = schedule["game_id"].astype(str).str.zfill(10)
    schedule["game_date"] = pd.to_datetime(schedule["game_date"], errors="coerce")

    schedule["home_tricode"] = schedule["home_tricode"].fillna(schedule["home_team_id"].map(TEAM_ID_TO_TRICODE))
    schedule["away_tricode"] = schedule["away_tricode"].fillna(schedule["away_team_id"].map(TEAM_ID_TO_TRICODE))

    return schedule


def compute_rest_features(schedule: pd.DataFrame) -> pd.DataFrame:
    """Compute rest days, back-to-back, and 3-in-4 features for each game."""
    schedule = schedule.dropna(subset=["game_date"]).copy()

    home = schedule[["game_id", "game_date", "home_tricode"]].rename(columns={"home_tricode": "tricode"})
    home["is_home"] = 1

    away = schedule[["game_id", "game_date", "away_tricode"]].rename(columns={"away_tricode": "tricode"})
    away["is_home"] = 0

    team_games = (
        pd.concat([home, away], ignore_index=True)
        .sort_values(["tricode", "game_date", "game_id"])
        .reset_index(drop=True)
    )

    team_games["prev_game_date"] = team_games.groupby("tricode")["game_date"].shift(1)
    team_games["rest_days"] = (team_games["game_date"] - team_games["prev_game_date"]).dt.days

    team_games["is_back_to_back"] = (team_games["rest_days"] == 1).astype(int)
    team_games["is_3_in_4"] = (team_games["rest_days"] <= 2).astype(int)

    home_rest = team_games[team_games["is_home"] == 1][
        ["game_id", "game_date", "rest_days", "is_back_to_back", "is_3_in_4"]
    ].copy()
    home_rest.columns = ["game_id", "game_date", "home_rest_days", "home_b2b", "home_3in4"]

    away_rest = team_games[team_games["is_home"] == 0][
        ["game_id", "rest_days", "is_back_to_back", "is_3_in_4"]
    ].copy()
    away_rest.columns = ["game_id", "away_rest_days", "away_b2b", "away_3in4"]

    rest = home_rest.merge(away_rest, on="game_id", how="inner")
    rest["game_id"] = rest["game_id"].astype(str).str.zfill(10)

    return rest


schedule = build_game_schedule_from_pbp(pbp)
rest_features = compute_rest_features(schedule)

rest_features.to_csv(REST_PATH, index=False)

print(f"Schedule: {schedule.shape}")
print(f"Rest features: {rest_features.shape}")
display(rest_features.head())

## 6. Team game stats and rolling strength features

Rolling features are shifted by one game, so each row only uses information available before the current game.

In [ ]:
def compute_game_team_stats(pbp: pd.DataFrame) -> pd.DataFrame:
    """Compute one row of team box-score-style stats per team per game."""
    rows = []

    for (game_id, tricode), grp in tqdm(pbp.groupby(["game_id", "teamTricode"]), desc="Team-game stats"):
        if pd.isna(tricode) or tricode == "":
            continue

        shots_2pt = grp[grp["actionType"] == "2pt"]
        shots_3pt = grp[grp["actionType"] == "3pt"]
        fts = grp[grp["actionType"] == "freethrow"]
        turnovers = grp[grp["actionType"] == "turnover"]
        rebounds = grp[grp["actionType"] == "rebound"]

        fgm_2 = (shots_2pt["shot_result"] == "Made").sum()
        fga_2 = len(shots_2pt)
        fgm_3 = (shots_3pt["shot_result"] == "Made").sum()
        fga_3 = len(shots_3pt)

        fgm = fgm_2 + fgm_3
        fga = fga_2 + fga_3
        ftm = (fts["shot_result"] == "Made").sum()
        fta = len(fts)
        tov = len(turnovers)

        rebound_desc = rebounds["description"].fillna("").str.lower()
        oreb = rebound_desc.str.contains("off:", regex=False).sum()
        dreb = rebound_desc.str.contains("def:", regex=False).sum()

        pts = fgm_2 * 2 + fgm_3 * 3 + ftm
        possessions = fga + 0.44 * fta + tov

        rows.append({
            "game_id": str(game_id).zfill(10),
            "tricode": tricode,
            "pts": pts,
            "fgm": fgm,
            "fga": fga,
            "fgm_3": fgm_3,
            "fga_3": fga_3,
            "ftm": ftm,
            "fta": fta,
            "tov": tov,
            "oreb": oreb,
            "dreb": dreb,
            "possessions": possessions,
            "efg": (fgm + 0.5 * fgm_3) / fga if fga > 0 else np.nan,
            "tov_pct": tov / possessions if possessions > 0 else np.nan,
            "ft_rate": fta / fga if fga > 0 else np.nan,
        })

    stats = pd.DataFrame(rows)

    opponent = stats[["game_id", "tricode", "pts", "possessions", "dreb", "oreb"]].copy()
    opponent.columns = ["game_id", "opp_tricode", "opp_pts", "opp_possessions", "opp_dreb", "opp_oreb"]

    stats = stats.merge(opponent, on="game_id", how="left")
    stats = stats[stats["tricode"] != stats["opp_tricode"]].copy()

    stats["oreb_pct"] = stats["oreb"] / (stats["oreb"] + stats["opp_dreb"]).replace(0, np.nan)
    stats["off_rtg"] = (stats["pts"] / stats["possessions"] * 100).replace([np.inf, -np.inf], np.nan)
    stats["def_rtg"] = (stats["opp_pts"] / stats["opp_possessions"] * 100).replace([np.inf, -np.inf], np.nan)
    stats["net_rtg"] = stats["off_rtg"] - stats["def_rtg"]
    stats["won"] = (stats["pts"] > stats["opp_pts"]).astype(int)

    return stats.reset_index(drop=True)


def add_rolling_team_features(team_stats: pd.DataFrame, schedule: pd.DataFrame) -> pd.DataFrame:
    """Add shifted rolling averages for last 10, last 20, and last season-ish windows."""
    team_dates = pd.concat([
        schedule[["game_id", "game_date", "home_tricode"]].rename(columns={"home_tricode": "tricode"}),
        schedule[["game_id", "game_date", "away_tricode"]].rename(columns={"away_tricode": "tricode"}),
    ], ignore_index=True)

    team_stats = team_stats.merge(team_dates, on=["game_id", "tricode"], how="left")
    team_stats["game_date"] = pd.to_datetime(team_stats["game_date"], errors="coerce")

    team_stats = team_stats.sort_values(["tricode", "game_date", "game_id"]).reset_index(drop=True)

    stat_cols = [
        "efg", "tov_pct", "ft_rate", "oreb_pct",
        "off_rtg", "def_rtg", "net_rtg", "won", "pts",
    ]

    windows = {
        "10": 10,
        "20": 20,
        "season": 82,
    }

    for label, window in windows.items():
        for col in stat_cols:
            team_stats[f"{col}_last{label}"] = (
                team_stats
                .groupby("tricode")[col]
                .transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
            )

    return team_stats


team_game_stats = compute_game_team_stats(pbp)
team_rolling_stats = add_rolling_team_features(team_game_stats, schedule)

team_rolling_stats.to_csv(TEAM_STATS_PATH, index=False)

print(f"Team rolling stats: {team_rolling_stats.shape}")
display(team_rolling_stats.head())

## 7. Join game-state rows with rolling team strength and rest features

In [ ]:
def prefix_team_stats(team_stats: pd.DataFrame, schedule: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Split team rolling stats into home and away feature tables."""
    home_map = schedule[["game_id", "home_tricode", "away_tricode"]].copy()
    home_map["game_id"] = home_map["game_id"].astype(str).str.zfill(10)

    team_stats = team_stats.copy()
    team_stats["game_id"] = team_stats["game_id"].astype(str).str.zfill(10)

    team_stats_full = team_stats.merge(home_map, on="game_id", how="left")

    rolling_cols = [
        c for c in team_stats.columns
        if any(c.endswith(suffix) for suffix in ["_last10", "_last20", "_lastseason"])
    ]

    id_cols = [
        "game_id", "tricode", "off_rtg", "def_rtg", "net_rtg",
        "efg", "tov_pct", "ft_rate", "oreb_pct", "won", "pts",
    ]

    keep_cols = [c for c in id_cols + rolling_cols if c in team_stats_full.columns]
    team_stats_full = team_stats_full[keep_cols + ["home_tricode", "away_tricode"]]

    home_stats = team_stats_full[team_stats_full["tricode"] == team_stats_full["home_tricode"]].copy()
    away_stats = team_stats_full[team_stats_full["tricode"] == team_stats_full["away_tricode"]].copy()

    home_stats = home_stats.drop(columns=["tricode", "home_tricode", "away_tricode"])
    away_stats = away_stats.drop(columns=["tricode", "home_tricode", "away_tricode"])

    home_stats = home_stats.rename(columns={c: f"home_{c}" for c in home_stats.columns if c != "game_id"})
    away_stats = away_stats.rename(columns={c: f"away_{c}" for c in away_stats.columns if c != "game_id"})

    return home_stats, away_stats


def add_differential_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add home-minus-away feature differences."""
    out = df.copy()

    diff_stats = ["off_rtg", "def_rtg", "net_rtg", "efg", "tov_pct", "ft_rate", "oreb_pct"]

    for window in ["last10", "last20", "lastseason"]:
        for stat in diff_stats:
            home_col = f"home_{stat}_{window}"
            away_col = f"away_{stat}_{window}"

            if home_col in out.columns and away_col in out.columns:
                out[f"diff_{stat}_{window}"] = out[home_col] - out[away_col]

    return out


def build_final_dataset(
    pbp: pd.DataFrame,
    team_rolling_stats: pd.DataFrame,
    rest_features: pd.DataFrame,
    schedule: pd.DataFrame,
) -> pd.DataFrame:
    """Join base play-by-play rows, team strength, rest, and date features."""
    home_stats, away_stats = prefix_team_stats(team_rolling_stats, schedule)

    final = pbp.copy()
    final["game_id"] = final["game_id"].astype(str).str.zfill(10)

    final = final.merge(home_stats, on="game_id", how="left")
    final = final.merge(away_stats, on="game_id", how="left")

    rest = rest_features.copy()
    rest["game_id"] = rest["game_id"].astype(str).str.zfill(10)

    rest_cols = [
        "game_id", "game_date",
        "home_rest_days", "home_b2b", "home_3in4",
        "away_rest_days", "away_b2b", "away_3in4",
    ]
    final = final.drop(columns=["game_date"], errors="ignore")
    final = final.merge(rest[rest_cols], on="game_id", how="left")

    final = add_differential_features(final)

    final["game_date"] = pd.to_datetime(final["game_date"], errors="coerce")
    final["day_of_week"] = final["game_date"].dt.dayofweek
    final["is_weekend"] = final["day_of_week"].isin([4, 5, 6]).astype(int)

    return final


df_final = build_final_dataset(pbp, team_rolling_stats, rest_features, schedule)
df_final.to_csv(FULL_OUTPUT_PATH, index=False)

print(f"Final dataset: {df_final.shape}")
print(f"Saved to: {FULL_OUTPUT_PATH}")

check_cols = [
    "home_net_rtg_last10", "away_net_rtg_last10",
    "home_rest_days", "away_rest_days",
    "diff_net_rtg_lastseason",
]
display(df_final[check_cols].isna().sum())
display(df_final.head())

## 8. Create slim modeling dataset

This file is much smaller and contains the columns most useful for the first live win-probability models.

In [ ]:
SLIM_COLUMNS = [
    "game_id", "is_playoffs", "home_team_won",
    "period", "is_overtime", "seconds_remaining", "score_diff",
    "momentum_5", "momentum_10", "home_possession", "actionType",
    "home_fouls", "away_fouls", "home_tech_fouls", "away_tech_fouls",
    "home_ejections", "away_ejections",
    "home_timeouts_used", "away_timeouts_used",
    "shot_distance", "shot_area", "shot_result", "is_field_goal",
    "shot_x", "shot_y", "is_fastbreak", "is_painttouch", "is_fromturnover",

    "home_net_rtg_last10", "home_net_rtg_last20", "home_net_rtg_lastseason",
    "home_off_rtg_last10", "home_def_rtg_last10",
    "home_efg_last10", "home_tov_pct_last10", "home_oreb_pct_last10",
    "home_off_rtg_lastseason", "home_def_rtg_lastseason",
    "home_won_last10", "home_won_lastseason",

    "away_net_rtg_last10", "away_net_rtg_last20", "away_net_rtg_lastseason",
    "away_off_rtg_last10", "away_def_rtg_last10",
    "away_efg_last10", "away_tov_pct_last10", "away_oreb_pct_last10",
    "away_off_rtg_lastseason", "away_def_rtg_lastseason",
    "away_won_last10", "away_won_lastseason",

    "game_date",
    "home_rest_days", "away_rest_days",
    "home_b2b", "away_b2b",
    "home_3in4", "away_3in4",

    "diff_net_rtg_last10", "diff_net_rtg_last20", "diff_net_rtg_lastseason",
    "diff_off_rtg_last10", "diff_def_rtg_last10",

    "day_of_week", "is_weekend",
]

available_slim_cols = [c for c in SLIM_COLUMNS if c in df_final.columns]
missing_slim_cols = [c for c in SLIM_COLUMNS if c not in df_final.columns]

df_slim = df_final[available_slim_cols].copy()
df_slim.to_csv(SLIM_OUTPUT_PATH, index=False)

print(f"Slim dataset: {df_slim.shape}")
print(f"Saved to: {SLIM_OUTPUT_PATH}")

if missing_slim_cols:
    print("Missing slim columns:")
    print(missing_slim_cols)

## 9. Quick validation checks

In [ ]:
def summarize_dataset(df: pd.DataFrame, name: str) -> None:
    print(f"{name}")
    print("-" * len(name))
    print(f"Shape: {df.shape}")
    print(f"Games: {df['game_id'].nunique():,}")
    print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

    if "game_date" in df.columns:
        dates = pd.to_datetime(df["game_date"], errors="coerce")
        print(f"Date range: {dates.min()} to {dates.max()}")

    if "home_team_won" in df.columns:
        game_level = df.drop_duplicates("game_id")
        print(f"Home win rate: {game_level['home_team_won'].mean():.3f}")


summarize_dataset(df_final, "Full enriched dataset")
print()
summarize_dataset(df_slim, "Slim modeling dataset")

print("\nAction types:")
display(df_slim["actionType"].value_counts().head(20))

print("\nNull check, key columns:")
key_cols = [
    "seconds_remaining", "score_diff", "home_team_won",
    "diff_net_rtg_lastseason", "home_rest_days", "away_rest_days",
]
key_cols = [c for c in key_cols if c in df_slim.columns]
display(df_slim[key_cols].isna().sum())

## 10. Optional: export a tiny sample for GitHub

The full CSV files are too large for normal GitHub commits. A small sample is useful for README examples and tests.

In [ ]:
SAMPLE_OUTPUT_PATH = PROCESSED_DIR / "nba_win_prob_slim_sample.csv"

sample_games = (
    df_slim
    .drop_duplicates("game_id")
    .sample(min(25, df_slim["game_id"].nunique()), random_state=42)["game_id"]
)

df_sample = df_slim[df_slim["game_id"].isin(sample_games)].copy()
df_sample.to_csv(SAMPLE_OUTPUT_PATH, index=False)

print(f"Sample dataset: {df_sample.shape}")
print(f"Saved to: {SAMPLE_OUTPUT_PATH}")

## Notes for modeling

The strongest first baseline so far is usually:

- `score_diff`
- `seconds_remaining`
- `period`
- `is_overtime`
- `home_possession`
- engineered score/time interactions
- `diff_net_rtg_lastseason`

When adding more features, evaluate with a temporal split and prioritize:

- log loss
- Brier score
- calibration
- clutch/close-game subset performance